# Pipeline de recertification KYC — Qwen2.5-VL uniquement

## Objectif
Extraire les informations des documents **JUSTIFICATIF IDENTITE.PDF** et
**JUSTIFICATIF DOMICILE.PDF** de chaque dossier client (archive ZIP, dossiers nommés par
l'ID client), puis comparer avec `tiers.csv` pour détecter les **incohérences**.

## Version de ce notebook
- **Un seul modèle : Qwen2.5-VL-7B-Instruct**, chargé via `AutoProcessor` +
  `AutoModelForVision2Seq` (au lieu de la classe dédiée `Qwen2_5_VLForConditionalGeneration`),
  en `float16`, avec `low_cpu_mem_usage=True` et `.to(DEVICE)` — reprend exactement le pattern
  de chargement que vous utilisez déjà dans votre notebook actuel.
- **Exécution progressive** : chaque section définit ses fonctions puis les **exécute
  immédiatement** sur les données réelles (client de test, puis boucle complète), plutôt que
  de tout empiler dans des cellules commentées en fin de notebook.

## Stratégie (inchangée sur le fond)
1. Dézipper l'archive, repérer les 2 PDF cibles par dossier client (recherche *fuzzy* du nom).
2. PDF → image haute résolution.
3. Prétraitement : correction de rotation (0/90/180/270), *deskew*, amélioration de contraste
   — utile car scans/photocopies de qualité très variable.
4. Détection du pays d'émission (Algérie vs autre) par mots-clés bilingues + confirmation Qwen.
5. Extraction :
   - **Algérie** → prompt avec **schéma JSON strict** par type de document (bilingue AR/FR).
   - **Hors Algérie** → prompt **libre** (JSON dynamique, pas de schéma imposé).
6. Parsing robuste du JSON renvoyé par le modèle.
7. Agrégation en DataFrame + rapprochement avec `tiers.csv` → rapport d'incohérences.


## 1. Imports

In [ ]:
import os
import re
import io
import json
import time
import zipfile
import shutil
import unicodedata
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

try:
    from rapidfuzz import fuzz
except ImportError:
    fuzz = None

try:
    import pytesseract
    HAS_TESSERACT = True
except ImportError:
    HAS_TESSERACT = False

print("Torch CUDA disponible :", torch.cuda.is_available())


## 2. Configuration

In [ ]:
# --- Modèle (ModelHub) ---
MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-instruct/main"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Données ---
ZIP_PATH = "/mnt/data/dossiers_clients.zip"          # archive ZIP des dossiers clients
WORKDIR = Path("/tmp/kyc_extraction")                # dossier de travail (extraction du zip)
TIERS_CSV_PATH = "/mnt/data/tiers.csv"                # référentiel tiers
OUTPUT_DIR = Path("/mnt/data/output_kyc")             # rapports de sortie

WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_DOCS = {
    "identite": "JUSTIFICATIF IDENTITE",
    "domicile": "JUSTIFICATIF DOMICILE",
}

DPI = 300
FUZZY_FILENAME_THRESHOLD = 80   # seuil de similarité (0-100) pour matcher un nom de fichier
MATCH_THRESHOLD_TIERS = 85.0    # seuil de similarité pour comparaison avec tiers.csv


## 3. Chargement du modèle Qwen2.5-VL

**C'est ici que se situe le changement demandé** par rapport à la version précédente : on
reprend exactement votre pattern de chargement (`AutoModelForVision2Seq`, `float16`,
`low_cpu_mem_usage=True`, `.to(DEVICE)`, mesure du temps de chargement), et il n'y a plus
qu'un seul modèle à gérer (plus de `load_model()`/cache multi-modèles, plus d'InternVL3 en
fallback, plus de PaddleOCR-VL).

> On ignore volontairement l'erreur `HFValidationError` visible sur votre capture — elle
> provient généralement d'un souci de résolution du chemin local par `trust_remote_code`
> (le code distant du repo tente un `hf_hub_download` au lieu de lire uniquement en local).
> Si elle réapparaît chez vous, la solution la plus fréquente est d'ajouter
> `local_files_only=True` sur les deux `from_pretrained`, ou de vérifier que `MODEL_PATH`
> pointe bien vers un dossier contenant tous les fichiers du repo (config, poids, tokenizer,
> preprocessor). Cela ne change rien à la structure du code ci-dessous.

In [ ]:
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16,
    trust_remote_code=True, low_cpu_mem_usage=True,
)
model.eval().to(DEVICE)

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')


## 4. Fonction générique d'appel au modèle Qwen

In [ ]:
def call_qwen(image: Image.Image, prompt: str, max_new_tokens: int = 1024,
              temperature: float = 0.0) -> str:
    \"\"\"Envoie une image + un prompt texte à Qwen2.5-VL et renvoie la réponse texte brute.\"\"\"
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    try:
        from qwen_vl_utils import process_vision_info
        image_inputs, video_inputs = process_vision_info(messages)
    except ImportError:
        image_inputs, video_inputs = [image], None

    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-5),
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return output_text.strip()


# Petit test de fumée pour valider que le modèle répond correctement
_test_img = Image.new("RGB", (300, 100), color="white")
print(call_qwen(_test_img, "Décris cette image en un mot."))


## 5. Dézippage et repérage des 2 PDF cibles par client

Exécution immédiate sur l'archive réelle : on construit l'index des clients et on affiche
un aperçu.

In [ ]:
def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip().upper()
    return s


def fuzzy_score(a: str, b: str) -> int:
    a, b = normalize_text(a), normalize_text(b)
    if fuzz is not None:
        return fuzz.partial_ratio(a, b)
    return 100 if b in a else 0


def extract_zip(zip_path: str, workdir: Path) -> Path:
    extract_to = workdir / "extracted"
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)
    return extract_to


def list_client_folders(extract_root: Path):
    folders = [p for p in extract_root.iterdir() if p.is_dir()]
    if not folders:
        folders = [p for p in extract_root.rglob("*") if p.is_dir()]
    return folders


def find_target_pdfs(client_folder: Path) -> dict:
    result = {"identite": None, "domicile": None}
    pdf_files = list(client_folder.rglob("*.pdf")) + list(client_folder.rglob("*.PDF"))
    for key, target_name in TARGET_DOCS.items():
        best_path, best_score = None, 0
        for pdf in pdf_files:
            score = fuzzy_score(pdf.stem, target_name)
            if score > best_score:
                best_score, best_path = score, pdf
        if best_score >= FUZZY_FILENAME_THRESHOLD:
            result[key] = best_path
    return result


def build_client_index(zip_path: str, workdir: Path) -> pd.DataFrame:
    extract_root = extract_zip(zip_path, workdir)
    folders = list_client_folders(extract_root)
    rows = []
    for folder in folders:
        client_id = folder.name
        targets = find_target_pdfs(folder)
        rows.append({
            "client_id": client_id,
            "path_identite": str(targets["identite"]) if targets["identite"] else None,
            "path_domicile": str(targets["domicile"]) if targets["domicile"] else None,
        })
    df = pd.DataFrame(rows)
    missing = df[df["path_identite"].isna() | df["path_domicile"].isna()]
    if len(missing):
        print(f"⚠️  {len(missing)} client(s) avec au moins un document cible manquant :")
        print(missing[["client_id"]].to_string(index=False))
    return df


client_index = build_client_index(ZIP_PATH, WORKDIR)
print(f"{len(client_index)} client(s) trouvé(s) dans l'archive.")
client_index.head()


## 6. PDF → image : test sur le premier client disponible

On prend le premier client dont les 2 documents sont présents, pour dérouler le pipeline
étape par étape sur un cas réel avant de lancer la boucle complète.

In [ ]:
import fitz  # PyMuPDF

def pdf_to_images(pdf_path: str, dpi: int = DPI):
    images = []
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    with fitz.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=mat, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            images.append(img)
    return images


def pick_main_page(images):
    if len(images) == 1:
        return images[0]
    scored = [(np.array(img.convert("L")).std(), img) for img in images]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


sample_row = client_index.dropna(subset=["path_identite", "path_domicile"]).iloc[0]
sample_client_id = sample_row["client_id"]
print("Client de test :", sample_client_id)

sample_identite_img = pick_main_page(pdf_to_images(sample_row["path_identite"]))
sample_domicile_img = pick_main_page(pdf_to_images(sample_row["path_domicile"]))
print("Taille image identité :", sample_identite_img.size)
print("Taille image domicile :", sample_domicile_img.size)
display(sample_identite_img.resize((400, int(400 * sample_identite_img.height / sample_identite_img.width))))


## 7. Prétraitement : rotation, redressement, amélioration — appliqué au cas de test

Correction de l'orientation (OSD Tesseract, fallback Qwen si l'OSD échoue), *deskew* léger,
et amélioration de contraste pour les photocopies pâles.

In [ ]:
def deskew(image: Image.Image) -> Image.Image:
    arr = np.array(image.convert("L"))
    arr = cv2.GaussianBlur(arr, (5, 5), 0)
    _, thresh = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 50:
        return image
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.5 or abs(angle) > 20:
        return image
    (h, w) = arr.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(np.array(image), M, (w, h),
                              flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(rotated)


def enhance_for_ocr(image: Image.Image) -> Image.Image:
    arr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    arr = clahe.apply(arr)
    arr = cv2.fastNlMeansDenoising(arr, h=10)
    return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB))


def detect_rotation_tesseract(image: Image.Image):
    if not HAS_TESSERACT:
        return None
    try:
        osd = pytesseract.image_to_osd(image)
        m = re.search(r"Rotate: (\d+)", osd)
        conf = re.search(r"Orientation confidence: ([\d.]+)", osd)
        if m and conf and float(conf.group(1)) >= 1.0:
            return int(m.group(1))
    except Exception:
        return None
    return None


def rotate_image(image: Image.Image, angle: int) -> Image.Image:
    if angle % 360 == 0:
        return image
    return image.rotate(-angle, expand=True)


def detect_rotation_qwen(image: Image.Image) -> int:
    prompt = (
        "Regarde ce document scanné (carte d'identité ou justificatif de domicile). "
        "Indique uniquement l'angle de rotation horaire nécessaire pour que le texte soit "
        "parfaitement à l'endroit et lisible. Réponds strictement par un seul nombre parmi "
        "0, 90, 180, 270."
    )
    answer = call_qwen(image, prompt)
    match = re.search(r"\b(0|90|180|270)\b", answer)
    return int(match.group(1)) if match else 0


def auto_orient_and_clean(image: Image.Image) -> Image.Image:
    angle = detect_rotation_tesseract(image)
    if angle is None:
        angle = detect_rotation_qwen(image)
    if angle:
        image = rotate_image(image, angle)
    image = deskew(image)
    image = enhance_for_ocr(image)
    return image


sample_identite_clean = auto_orient_and_clean(sample_identite_img)
sample_domicile_clean = auto_orient_and_clean(sample_domicile_img)
print("Prétraitement appliqué sur les 2 documents du client de test.")
display(sample_identite_clean.resize((400, int(400 * sample_identite_clean.height / sample_identite_clean.width))))


## 8. Détection du pays d'émission — testée sur le cas réel

In [ ]:
DZ_KEYWORDS = [
    "REPUBLIQUE ALGERIENNE", "REPUBLIQUE ALGERIENNE DEMOCRATIQUE ET POPULAIRE",
    "الجمهورية الجزائرية", "وزارة الداخلية", "BLIDA", "ALGER", "ORAN", "CONSTANTINE",
    "WILAYA", "COMMUNE DE", "الجزائر", "بطاقة التعريف الوطنية", "CARTE NATIONALE D'IDENTITE",
]

def heuristic_country_score(raw_text: str) -> float:
    text_norm = normalize_text(raw_text)
    hits = sum(1 for kw in DZ_KEYWORDS if normalize_text(kw) in text_norm)
    return min(hits / 2.0, 1.0)


def detect_country(image: Image.Image, raw_ocr_text: str = "") -> str:
    score = heuristic_country_score(raw_ocr_text)
    if score >= 0.5:
        return "DZ"
    if raw_ocr_text and score == 0.0:
        return "OTHER"
    prompt = (
        "Ce document est-il une pièce d'identité ou un justificatif de domicile émis en "
        "Algérie ? Réponds strictement par 'DZ' si oui, ou 'OTHER' si le document provient "
        "d'un autre pays."
    )
    answer = call_qwen(image, prompt)
    return "DZ" if "DZ" in answer.upper() else "OTHER"


sample_country = detect_country(sample_identite_clean)
print("Pays détecté pour le client de test :", sample_country)


## 9. Schémas d'extraction pour les documents émis en Algérie

In [ ]:
SCHEMA_DZ_IDENTITE = {
    "type_document": "carte_identite_nationale | passeport | permis_conduire",
    "nom": "string (nom de famille, اللقب)",
    "prenom": "string (prénom, الاسم)",
    "date_naissance": "YYYY-MM-DD",
    "lieu_naissance": "string",
    "sexe": "M | F",
    "numero_piece": "string (numéro de la carte / passeport)",
    "date_delivrance": "YYYY-MM-DD",
    "date_expiration": "YYYY-MM-DD",
    "autorite_delivrance": "string (wilaya / commune / daira)",
    "adresse": "string (adresse figurant sur la pièce, si présente)",
}

SCHEMA_DZ_DOMICILE = {
    "type_document": "facture_sonelgaz | facture_ADE | attestation_communale | autre",
    "nom_titulaire": "string",
    "prenom_titulaire": "string",
    "adresse_complete": "string",
    "commune": "string",
    "wilaya": "string",
    "code_postal": "string ou null",
    "date_document": "YYYY-MM-DD",
    "numero_reference": "string ou null (n° facture / n° acte)",
}

def build_schema_prompt(schema: dict, doc_label: str) -> str:
    schema_json = json.dumps(schema, ensure_ascii=False, indent=2)
    return f\"\"\"Tu es un expert en extraction de données KYC. Le document ci-joint est un
{doc_label} émis en Algérie. Le texte peut être en arabe, en français, ou bilingue.

Extrait les informations selon EXACTEMENT le schéma JSON suivant (mêmes clés). Si une
information est absente ou illisible, mets la valeur null. Ne complète jamais un champ par
une supposition. Normalise les dates au format YYYY-MM-DD.

Schéma attendu :
{schema_json}

Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ni après, sans balises
markdown.\"\"\"


## 10. Prompt d'extraction générique pour les documents hors Algérie

In [ ]:
def build_generic_prompt(doc_label: str) -> str:
    return f\"\"\"Tu es un expert en extraction de données KYC. Le document ci-joint est un
{doc_label} émis dans un pays autre que l'Algérie (format inconnu à l'avance).

Identifie le pays d'émission et le type précis de document, puis extrait toutes les
informations pertinentes que tu peux lire avec certitude (nom, prénom, date de naissance,
numéro de pièce, dates de délivrance/expiration, adresse complète, autorité émettrice, etc.).

Réponds UNIQUEMENT avec un objet JSON de la forme :
{{
  "pays_detecte": "...",
  "type_document": "...",
  "champs": {{ "nom_du_champ": "valeur", ... }}
}}

N'invente aucune valeur : si un champ est illisible ou absent, ne l'inclus pas. Normalise les
dates au format YYYY-MM-DD quand c'est possible. Pas de texte hors du JSON, pas de markdown.\"\"\"


## 11. Parsing robuste du JSON renvoyé par Qwen

In [ ]:
def parse_json_safe(raw_output: str):
    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```json\s*|\s*```$", "", cleaned, flags=re.MULTILINE)
    cleaned = re.sub(r"^```\s*|\s*```$", "", cleaned, flags=re.MULTILINE)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None


## 12. Orchestration de l'extraction pour un document — testée sur le cas réel

Pipeline complet pour un document donné : prétraitement (déjà fait ci-dessus pour le cas de
test) → choix du prompt (schéma DZ ou libre selon le pays) → appel Qwen → parsing JSON, avec
une nouvelle tentative si le JSON est invalide.

In [ ]:
def extract_document(clean_image: Image.Image, doc_type: str, country: str) -> dict:
    \"\"\"doc_type: 'identite' ou 'domicile' ; country: 'DZ' ou 'OTHER'\"\"\"
    result = {
        "doc_type": doc_type, "pays_detecte": country,
        "extraction_ok": False, "data": None, "erreur": None,
    }
    doc_label = "carte d'identité / pièce d'identité" if doc_type == "identite" else \
                "justificatif de domicile"

    if country == "DZ":
        schema = SCHEMA_DZ_IDENTITE if doc_type == "identite" else SCHEMA_DZ_DOMICILE
        prompt = build_schema_prompt(schema, doc_label)
    else:
        prompt = build_generic_prompt(doc_label)

    for attempt in range(2):
        raw_output = call_qwen(clean_image, prompt)
        parsed = parse_json_safe(raw_output)
        if parsed:
            result["data"] = parsed
            result["extraction_ok"] = True
            break
        result["erreur"] = f"JSON invalide (tentative {attempt+1}) : {raw_output[:200]}"

    return result


# Exécution sur le client de test
sample_result_identite = extract_document(sample_identite_clean, "identite", sample_country)
sample_result_domicile = extract_document(sample_domicile_clean, "domicile", sample_country)

print("--- Résultat identité ---")
print(json.dumps(sample_result_identite, ensure_ascii=False, indent=2))
print("--- Résultat domicile ---")
print(json.dumps(sample_result_domicile, ensure_ascii=False, indent=2))


## 13. Boucle complète sur tous les clients de l'archive

Une fois le pipeline validé sur le cas de test ci-dessus, on l'exécute sur l'ensemble des
clients trouvés dans l'index (section 5), avec un checkpoint incrémental.

In [ ]:
def process_one_document(pdf_path: str, doc_type: str) -> dict:
    if not pdf_path:
        return {"doc_type": doc_type, "extraction_ok": False, "erreur": "Fichier absent",
                "pays_detecte": None, "data": None}
    try:
        img = pick_main_page(pdf_to_images(pdf_path))
        clean_img = auto_orient_and_clean(img)
        country = detect_country(clean_img)
        return extract_document(clean_img, doc_type, country)
    except Exception as e:
        return {"doc_type": doc_type, "extraction_ok": False, "erreur": str(e),
                "pays_detecte": None, "data": None}


def flatten_results(records) -> pd.DataFrame:
    rows = []
    for entry in records:
        row = {"client_id": entry["client_id"]}
        for doc_type in ["identite", "domicile"]:
            res = entry.get(f"{doc_type}_result", {})
            row[f"{doc_type}_ok"] = res.get("extraction_ok", False)
            row[f"{doc_type}_pays"] = res.get("pays_detecte")
            data = res.get("data") or {}
            if "champs" in data:
                flat = {**{k: v for k, v in data.items() if k != "champs"}, **data["champs"]}
            else:
                flat = data
            for k, v in flat.items():
                row[f"{doc_type}__{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)


checkpoint_every = 20
checkpoint_path = OUTPUT_DIR / "extraction_checkpoint.jsonl"
if checkpoint_path.exists():
    checkpoint_path.unlink()

records = []
for i, row in client_index.iterrows():
    entry = {
        "client_id": row["client_id"],
        "identite_result": process_one_document(row["path_identite"], "identite"),
        "domicile_result": process_one_document(row["path_domicile"], "domicile"),
    }
    records.append(entry)

    if (i + 1) % checkpoint_every == 0:
        with open(checkpoint_path, "a", encoding="utf-8") as f:
            for r in records[-checkpoint_every:]:
                f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
        print(f"  ... {i+1}/{len(client_index)} clients traités")

remaining = len(records) % checkpoint_every
if remaining:
    with open(checkpoint_path, "a", encoding="utf-8") as f:
        for r in records[-remaining:]:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")

extraction_df = flatten_results(records)
extraction_df.to_csv(OUTPUT_DIR / "extraction_resultats.csv", index=False)
print(f"✅ Extraction terminée : {len(extraction_df)} clients traités.")
extraction_df.head()


## 14. Rapprochement avec `tiers.csv` et détection des incohérences

Exécuté directement sur `extraction_df` obtenu ci-dessus.

> ⚠️ Adapter `FIELD_MAPPING` aux noms de colonnes réels de votre `tiers.csv`.

In [ ]:
def load_tiers(tiers_csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(tiers_csv_path, dtype=str)
    df.columns = [c.strip().lower() for c in df.columns]
    return df


FIELD_MAPPING = {
    "identite__nom": "nom",
    "identite__prenom": "prenom",
    "identite__date_naissance": "date_naissance",
    "identite__numero_piece": "numero_piece_identite",
    "domicile__adresse_complete": "adresse",
}


def norm_compare(a, b) -> float:
    a, b = normalize_text(str(a)), normalize_text(str(b))
    if not a or not b or a == "NONE" or b == "NONE":
        return 0.0
    if fuzz is not None:
        return fuzz.token_sort_ratio(a, b)
    return 100.0 if a == b else 0.0


def build_incoherence_report(extraction_df: pd.DataFrame, tiers_df: pd.DataFrame,
                              match_threshold: float = MATCH_THRESHOLD_TIERS) -> pd.DataFrame:
    merged = extraction_df.merge(
        tiers_df, left_on="client_id", right_on="client_id", how="left",
        suffixes=("", "_tiers")
    )
    report_rows = []
    for _, row in merged.iterrows():
        incoherences = []
        for extr_col, tiers_col in FIELD_MAPPING.items():
            if extr_col not in row or tiers_col not in row:
                continue
            score = norm_compare(row.get(extr_col), row.get(tiers_col))
            if score < match_threshold:
                incoherences.append({
                    "champ": extr_col,
                    "valeur_extraite": row.get(extr_col),
                    "valeur_tiers": row.get(tiers_col),
                    "score_similarite": round(score, 1),
                })
        report_rows.append({
            "client_id": row["client_id"],
            "identite_extraction_ok": row.get("identite_ok"),
            "domicile_extraction_ok": row.get("domicile_ok"),
            "nb_incoherences": len(incoherences),
            "incoherences": incoherences,
        })
    return pd.DataFrame(report_rows).sort_values("nb_incoherences", ascending=False)


tiers_df = load_tiers(TIERS_CSV_PATH)
incoherence_report = build_incoherence_report(extraction_df, tiers_df)
print(f"{(incoherence_report['nb_incoherences'] > 0).sum()} client(s) avec au moins une incohérence.")
incoherence_report.head(20)


## 15. Export du rapport final

In [ ]:
def export_final_report(extraction_df: pd.DataFrame, incoherence_report: pd.DataFrame,
                          output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    xlsx_path = output_dir / f"rapport_kyc_{timestamp}.xlsx"

    detail_rows = []
    for _, r in incoherence_report.iterrows():
        for inc in r["incoherences"]:
            detail_rows.append({"client_id": r["client_id"], **inc})
    detail_df = pd.DataFrame(detail_rows)

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        extraction_df.to_excel(writer, sheet_name="extraction_brute", index=False)
        incoherence_report.drop(columns=["incoherences"]).to_excel(
            writer, sheet_name="synthese_incoherences", index=False
        )
        detail_df.to_excel(writer, sheet_name="detail_incoherences", index=False)

    print(f"✅ Rapport exporté : {xlsx_path}")
    return xlsx_path


rapport_path = export_final_report(extraction_df, incoherence_report, OUTPUT_DIR)


## 16. Notes & recommandations

- **Revue humaine ciblée** : utiliser `nb_incoherences > 0` comme file de contrôle manuel, pas
  comme rejet automatique.
- **Seuils** : `FUZZY_FILENAME_THRESHOLD` et `MATCH_THRESHOLD_TIERS` sont à calibrer sur un
  échantillon réel avant mise en production.
- **`AutoModelForVision2Seq`** : si cette classe ne reconnaît pas l'architecture Qwen2.5-VL
  dans votre version de `transformers`, la classe dédiée `Qwen2_5_VLForConditionalGeneration`
  (ou `AutoModelForImageTextToText` selon la version) reste l'alternative la plus fiable — le
  reste du pipeline (`call_qwen`, prompts, parsing) ne change pas.
- **Un seul modèle chargé** = empreinte VRAM plus prévisible, mais plus de filet de sécurité
  (pas de second modèle en cas d'échec JSON répété) — le mécanisme de 2 tentatives dans
  `extract_document` reste le seul garde-fou côté extraction.
- **Traçabilité** : chaque enregistrement conserve `pays_detecte` et le contenu brut en cas
  d'erreur, utile pour l'audit du processus.
